In [ ]:
import requests
import pandas as pd
import re

def fetch_uniprot_excel(query, fields=None, output_file="uniprot_results.xlsx"):
    """
    Fetch protein data from UniProt (all pages) and save as Excel with percentage progress updates.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"

    params = {
        "query": query,
        "format": "tsv",
        "size": 500
    }

    if fields:
        params["fields"] = ",".join(fields)

    all_data = []
    header = None
    url = base_url
    total = None  # total number of hits (if available)

    while True:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"❌ Error {response.status_code}: {response.text}")
            break

        # UniProt gives total count in the "X-Total-Results" header
        if total is None and "X-Total-Results" in response.headers:
            total = int(response.headers["X-Total-Results"])

        lines = response.text.splitlines()

        # Store header only once
        if header is None:
            header = lines[0].split("\t")

        # Append rows
        all_data.extend([row.split("\t") for row in lines[1:]])

        # Print percentage progress
        if total:
            percent = (len(all_data) / total) * 100
            print(f"📊 Progress: {percent:.2f}%")

        # Pagination: look for "next" URL
        if "Link" in response.headers:
            match = re.search(r'<(https?://[^>]+)>;\s*rel="next"', response.headers["Link"])
            if match:
                url = match.group(1)   # full absolute URL
                params = {}            # already included in next_url
                continue

        break  # no more pages

    # Convert to DataFrame
    df = pd.DataFrame(all_data, columns=header)
    df.to_excel(output_file, index=False)
    print(f"✅ Finished: 100% ({len(df)} entries). Saved to {output_file}")




In [ ]:
query="heme binding protein"
fields=["id", "organism_name", "gene_names","protein_name" ,"length","xref_pdb","xref_alphafolddb","Sequence"]
output_file= "keyword_search.xlsx"

In [ ]:
fetch_uniprot_excel(
    query=query,
    fields=fields,
    output_file= output_file
)

In [ ]:
import pandas as pd

excel_file_path = "/content/keyword_search.xlsx"
df = pd.read_excel(excel_file_path)


In [ ]:
df = df.drop('Sequence', axis=1)


In [ ]:
df.shape

(392157, 7)

In [ ]:
import numpy as np
# Create the new column '3d_structure'
# It is True if either 'PDB' or 'AlphaFoldDB' is not NaN, otherwise False
df['3d_structure'] = df.apply(lambda row: not (pd.isna(row['PDB']) and pd.isna(row['AlphaFoldDB'])), axis=1)

In [ ]:
df = df.drop(['PDB','AlphaFoldDB'], axis=1)

In [ ]:
def search_keyword_in_columns(df, query_string, columns_to_search):
    """
    Searches for keywords from a query string (separated by 'OR') in specified
    columns of a DataFrame and creates a new column indicating which column
    contains any of the keywords.

    Args:
        df (pd.DataFrame): The input DataFrame.
        query_string (str): The query string containing keywords separated by 'OR'.
        columns_to_search (list): A list of column names to search within.

    Returns:
        pd.DataFrame: The DataFrame with the added 'keyword_found_in' column.
    """
    queries = [q.strip() for q in query_string.split(" OR ")] # Split the query string by 'OR' and remove whitespace
    df['keyword_found_in'] = ''
    for index, row in df.iterrows():
        found_columns = []
        for col in columns_to_search:
            if col in row and pd.notna(row[col]):
                cell_content = str(row[col]).lower()
                for query in queries:
                    if query.lower() in cell_content:
                        found_columns.append(col)
                        break # Stop searching in this cell once a keyword is found
        if found_columns:
            df.loc[index, 'keyword_found_in'] = ', '.join(found_columns)
        else:
            df.loc[index, 'keyword_found_in'] = 'Not Found'
    return df

# Define the query string and the columns to search within
search_query = query # Use the original query variable
search_columns = ["Entry Name", "Organism", "Gene Names", "Protein names"] # Specify relevant columns

# Apply the function to your DataFrame
df = search_keyword_in_columns(df, search_query, search_columns)

# Display the updated DataFrame with the new column
display(df.head())

,Entry Name,Organism,Gene Names,Protein names,Length,3d_structure,keyword_found_in
0,PQIB_ECOLI,Escherichia coli (strain K12),pqiB pqi5B b0951 JW0934,Intermembrane transport protein PqiB (Paraquat...,546,True,"Entry Name, Gene Names, Protein names"
1,MLAD_ECOLI,Escherichia coli (strain K12),mlaD yrbD b3193 JW3160,Intermembrane phospholipid transport system bi...,183,True,"Entry Name, Gene Names, Protein names"
2,LETB_ECOLI,Escherichia coli (strain K12),letB yebT b1834 JW1823,Lipophilic envelope-spanning tunnel protein B ...,877,True,"Gene Names, Protein names"
3,MLAB_ECOLI,Escherichia coli (strain K12),mlaB yrbB b3191 JW5535,Intermembrane phospholipid transport system bi...,97,True,"Entry Name, Gene Names, Protein names"
4,MLAF_ECOLI,Escherichia coli (strain K12),mlaF yrbF b3195 JW3162,Intermembrane phospholipid transport system AT...,269,True,"Entry Name, Gene Names, Protein names"


In [ ]:
df_sorted = df.sort_values(by='Organism')

# Display the first few rows of the sorted DataFrame
display(df_sorted.head())

,Entry Name,Organism,Gene Names,Protein names,Length,3d_structure,keyword_found_in
80158,A0A1Y5H7I8_UNCXX,'Osedax' symbiont bacterium Rs2_46_30_T18,A9R01_10745,Toluene tolerance protein,204,True,Not Found
199044,A0A1Y5GKJ3_UNCXX,'Osedax' symbiont bacterium Rs2_46_30_T18,A9R01_16510,STAS domain-containing protein,94,True,Not Found
60654,A0A1Y5HCK5_UNCXX,'Osedax' symbiont bacterium Rs2_46_30_T18,A9R01_10750,Outer membrane lipid asymmetry maintenance pro...,151,True,Protein names
113102,A0A1Y5HFP8_UNCXX,'Osedax' symbiont bacterium Rs2_46_30_T18,A9R01_06225,Paraquat-inducible membrane protein A,164,True,Not Found
61342,A0A1Y5H7G8_UNCXX,'Osedax' symbiont bacterium Rs2_46_30_T18,A9R01_10760,Phospholipid ABC transporter ATP-binding prote...,266,True,Protein names


In [ ]:
# Rename columns to match the desired output format
df_formatted = df.rename(columns={
    'Entry Name': 'UniProt\nid',
    'Organism': 'Organism\nname',
    'Gene Names': 'Gene\nname',
    'Protein names': 'Protein\nname',
    'Length': 'Protein\nlength',
    '3d_structure': '3D\nstructure\navailable\n(Y/N)',
    'keyword_found_in': 'Field in which the\nkeyword is\npresent'
})

# Convert '3D structure available (Y/N)' column to 'Y' or 'N'
df_formatted['3D\nstructure\navailable\n(Y/N)'] = df_formatted['3D\nstructure\navailable\n(Y/N)'].apply(lambda x: 'Y' if x else 'N')

# Add 'S. No.' column
df_formatted.insert(0, 'S.\nNo.', range(1, 1 + len(df_formatted)))

# Reorder columns to match the desired output format
desired_order = [
    'S.\nNo.',
    'UniProt\nid',
    'Organism\nname',
    'Gene\nname',
    'Protein\nname',
    'Protein\nlength',
    '3D\nstructure\navailable\n(Y/N)',
    'Field in which the\nkeyword is\npresent'
]
df_formatted = df_formatted[desired_order]

# Display the formatted DataFrame
display(df_formatted)

,S.\nNo.,UniProt\nid,Organism\nname,Gene\nname,Protein\nname,Protein\nlength,3D\nstructure\navailable\n(Y/N),Field in which the\nkeyword is\npresent
0,1,PQIB_ECOLI,Escherichia coli (strain K12),pqiB pqi5B b0951 JW0934,Intermembrane transport protein PqiB (Paraquat...,546,Y,"Entry Name, Gene Names, Protein names"
1,2,MLAD_ECOLI,Escherichia coli (strain K12),mlaD yrbD b3193 JW3160,Intermembrane phospholipid transport system bi...,183,Y,"Entry Name, Gene Names, Protein names"
2,3,LETB_ECOLI,Escherichia coli (strain K12),letB yebT b1834 JW1823,Lipophilic envelope-spanning tunnel protein B ...,877,Y,"Gene Names, Protein names"
3,4,MLAB_ECOLI,Escherichia coli (strain K12),mlaB yrbB b3191 JW5535,Intermembrane phospholipid transport system bi...,97,Y,"Entry Name, Gene Names, Protein names"
4,5,MLAF_ECOLI,Escherichia coli (strain K12),mlaF yrbF b3195 JW3162,Intermembrane phospholipid transport system AT...,269,Y,"Entry Name, Gene Names, Protein names"
...,...,...,...,...,...,...,...,...
392152,392153,A0A069QMP7_HOYLO,Hoylesella loescheii DSM 19665 = JCM 12249 = A...,HMPREF1991_02813,Sigma-70 region 2,570,Y,Not Found
392153,392154,A0AB37MP59_9BACE,Bacteroides sp. AF16-7,DWW72_11770,RNA polymerase subunit sigma-70,546,N,Not Found
392154,392155,A0AB37N4B7_9BACE,Bacteroides sp. AF04-22,DWV31_03620,Sigma-70 family RNA polymerase sigma factor,546,N,Not Found
392155,392156,I0TDG1_9BACT,Prevotella sp. oral taxon 306 str. F0472,HMPREF9969_2480,ECF sigma factor,579,Y,Not Found


In [ ]:
df_formatted.shape

(392157, 8)

In [ ]:
output_excel_file = "/content/uniprot_results.xlsx"
df_formatted.to_excel(output_excel_file, index=False)
print(f"Formatted data saved to {output_excel_file}")

Formatted data saved to /content/uniprot_results.xlsx
